In [9]:
# Load env variables
from dotenv import load_dotenv

load_dotenv()

# Create an API client
from anthropic import Anthropic

client = Anthropic()
model = "claude-sonnet-5"

# Helper functions
def add_user_message(messages, text):
    user_message = ({"role": "user", "content": text})
    messages.append(user_message)

def add_assistant_message(messages, text):
    assistant_message = ({"role": "assistant", "content": text})
    messages.append(assistant_message)

def chat(messages, system=None, effort=None, stop_sequences=None):
    params = {
        "model": model,
        "max_tokens": 1000,
        "messages": messages,
    }

    if system:
        params["system"] = system

    if effort:
        params["output_config"] = {"effort": effort}

    if stop_sequences:
        params["stop_sequences"] = stop_sequences

    message = client.messages.create(**params)
    # Find the text block instead of assuming content[0] is text
    for block in message.content:
        if block.type == "text":
            return block.text

    return None  # fallback if no text block was found

In [ ]:
messages = []

add_user_message(messages, "Generate a very short event bridge rule as json")
add_assistant_message(messages, "```json")

chat(messages, stop_sequences=["```"])

Newer models no longer support this: Anthropic removed support for assistant message prefilling — any request where the messages array ends with role "assistant" now returns a 400 error.

What to use instead: the official replacement is a system-prompt instruction rather than a fake assistant turn

In [10]:
messages = []

add_user_message(messages,"Generate a very short event bridge rule as json")
answer = chat(messages, system="Respond with ONLY the JSON object — no markdown code fences, no explanation, no preamble.")

answer

'```json\n{\n  "source": ["aws.ec2"],\n  "detail-type": ["EC2 Instance State-change Notification"],\n  "detail": {\n    "state": ["running"]\n  }\n}\n```'

In [12]:
import json

cleaned = answer.strip()

if cleaned.startswith("```"):
    lines = cleaned.splitlines()
    lines = lines[1:]
    if lines and lines[-1].strip() == "```":
        lines = lines[:-1]
    cleaned = "\n".join(lines).strip()

json.loads(cleaned)

{'source': ['aws.ec2'],
 'detail-type': ['EC2 Instance State-change Notification'],
 'detail': {'state': ['running']}}

Use message prefilling and stop seqeunces only to get three different commands in a single response  
There shouldn't be any comments or explanation  
Hint: message prefilling isn't limited to just characters like ```

In [21]:
messages = []

prompt = """
Generate three different sample AWS CLI commands. Each should be very short.
"""

add_user_message(messages, prompt)

text = chat(messages, system="Respond with ONLY the three commands, with no explanation, no preamble.")
text.strip()

'```\naws s3 ls\naws ec2 describe-instances\naws iam list-users\n```'

In [22]:
from IPython.display import Markdown

Markdown(text)

```
aws s3 ls
aws ec2 describe-instances
aws iam list-users
```